<a href="https://colab.research.google.com/github/JulTob/Modelling/blob/main/GLM/GLM_2_07_Exploracion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07-Cox · Exploración gráfica de la cartera

🏵 **Mirar, formular una pregunta y comprobarla.** Una fila representa una póliza; no todas han estado observadas durante el mismo tiempo. Este cuaderno reúne la exploración gráfica en R y conserva la carga, los primeros histogramas y el cálculo de tasas del cuaderno de partida.

## Ruta de lectura

1. **Qué tenemos:** carga, tipos de variables y datos ausentes.
2. **Qué forma tienen los datos:** histogramas, curva normal de referencia y gráficos Q–Q.
3. **Qué relaciones aparecen:** dispersión, matriz de pares, cajas, barras y mapas de calor.
4. **Qué significa contar durante tiempos distintos:** exposición, violín y tasas.

Cada sección contiene una pregunta, una técnica y una guía para interpretar su salida. Puedes ejecutar el cuaderno completo o detenerte en cada pregunta. Los bloques de costes, bonus-malus y tiempos completan la exploración del conjunto.

**Datos:** cartera del equipo 07-Cox, fijada al commit `698dad06c4228ea79319fdba27647a0c61474703`. Sus 3062 filas y 40 campos se comprueban al cargar. Los valores descriptivos escritos en el texto se refieren a esa versión.

**Verificación de esta incorporación:** estructura del notebook y cifras comprobadas; código R revisado, pero no ejecutado en el entorno de preparación. Ejecuta las celdas de arriba abajo en Colab para producir sus salidas. La primera celda instala únicamente los paquetes que falten.

## 0. Cargar la cartera

Se conserva la fuente fijada que ya estabas usando. `cartera` mantiene sus 40 columnas originales. La conversión de exposición a años se hará en `data`, para que ambas unidades queden identificadas.

equipo	N	m_sin	disp_sin	p0_fraude	p_evento	p_baja	m_coste
equipo_07	260107	1.1	1.1	1	1.1	0.7	1.7	1.05	0.56


In [ ]:
# Preparar las herramientas que usa este cuaderno.
paquetes <- c("tidyverse", "treemapify")
faltantes <- paquetes[!vapply(paquetes, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes) > 0) {
  install.packages(faltantes, repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(treemapify))

# Fuente de datos del equipo 07: la misma versión del cuaderno de partida.
repositorio <- "https://raw.githubusercontent.com/JulTob/Modelling/698dad06c4228ea79319fdba27647a0c61474703/GLM/"
cartera <- read_csv(paste0(repositorio, "cartera.csv"), show_col_types = FALSE)

stopifnot(
  nrow(cartera) == 3062L, ncol(cartera) == 40L,
  all(c("id_poliza", "n_siniestros", "exposicion", "antiguedad") %in% names(cartera)),
  !anyDuplicated(cartera$id_poliza),
  all(is.finite(cartera$exposicion)), all(cartera$exposicion > 0),
  all(abs(cartera$exposicion * 10 - cartera$antiguedad) < 1e-8)
)

# Apariencia común; cada gráfico conserva sus unidades y su propia pregunta.
options(repr.plot.width = 10, repr.plot.height = 6)
theme_set(theme_minimal(base_size = 12))
glimpse(cartera)
summary(cartera$exposicion)

## 1. Qué representa cada columna

🔰 **El tipo de almacenamiento no determina el significado.** `id_agencia` es un código aunque R lo almacene como número. `bonus_malus_act` es una clase ordenada. `n_siniestros` es un conteo. `exposicion` es tiempo acumulado, expresado en décadas de póliza.

El inventario siguiente recorre los **40 campos** y muestra su familia, clase en R, número de valores distintos y ausencias. Si falta un nombre, la ejecución se detiene: así un error de nombre no hace desaparecer un gráfico silenciosamente.

💡 Los nombres reales son `zona_circulacion`, `tipo_vehiculo` y `color_vehiculo`.

In [ ]:
# Equipo 07-Cox. Primera celda: los 40 campos, por significado.
campos_por_familia <- list(
  identificadores = c("id_poliza", "id_agencia", "id_region"),
  exposicion = c("antiguedad", "exposicion"),
  medidas = c("edad_conductor", "antiguedad_carnet", "potencia_cv", "cilindrada",
              "antiguedad_vehiculo", "valor_vehiculo", "densidad", "km_anuales"),
  categorias = c("edad_tramo", "zona_circulacion", "uso", "tipo_vehiculo",
                 "combustible", "sexo", "color_vehiculo", "estado_civil",
                 "forma_pago", "financiado", "nivel_estudios", "medio_contacto",
                 "bonus_malus_prev", "bonus_malus_act"),
  conteos = c("n_conductores", "n_siniestros", "n_asistencia", "n_fraude", "n_gestiones"),
  tiempos_y_estados = c("t_primer_sin", "evento", "t_baja", "baja",
                       "motivo_fin", "tuvo_siniestro"),
  costes = c("coste_total", "coste_medio")
)
campos <- unlist(campos_por_familia, use.names = FALSE)
stopifnot(length(campos) == 40, !anyDuplicated(campos))
faltan <- setdiff(campos, names(cartera))
sobran <- setdiff(names(cartera), campos)
if (length(faltan) || length(sobran)) {
  stop("Revisar inventario. Faltan: ", paste(faltan, collapse = ", "),
       "; fuera del inventario: ", paste(sobran, collapse = ", "))
}
inventario <- tibble(
  familia = rep(names(campos_por_familia), lengths(campos_por_familia)),
  variable = campos,
  clase_R = vapply(cartera[campos], function(x) paste(class(x), collapse = "/"), character(1)),
  distintos = vapply(cartera[campos], function(x) n_distinct(x, na.rm = TRUE), integer(1)),
  ausentes = vapply(cartera[campos], function(x) sum(is.na(x)), integer(1))
)
print(inventario, n = Inf)
# La clase R es de almacenamiento; la familia expresa el papel de la variable.
# Consulta de una póliza: cambiar solo el identificador.
# cartera %>% filter(id_poliza == 1) %>% glimpse()

### Datos ausentes: una pregunta antes de limpiar

No borres todas las filas incompletas de la cartera. En esta versión, los **2113 `NA` de `coste_medio` coinciden exactamente con las pólizas sin siniestros**. El cociente coste total / número de siniestros queda indefinido cuando el denominador es cero. Esas pólizas siguen aportando información a los conteos y a la exposición.

La comprobación cruzada distingue un dato no definido de un coste total igual a cero.

In [ ]:
inventario %>% filter(ausentes > 0) %>% select(variable, ausentes) %>% print()
cartera %>%
  count(sin_siniestros = n_siniestros == 0,
        coste_medio_ausente = is.na(coste_medio)) %>% print()
stopifnot(all(is.na(cartera$coste_medio) == (cartera$n_siniestros == 0)))

## 2. Histogramas: concentración, colas y límites

**Pregunta:** ¿dónde se concentran las pólizas y hasta dónde llegan los valores?

El histograma divide el eje horizontal en intervalos y cuenta cuántos datos caen dentro. Conservamos tu histograma de antigüedad y el de exposición. En esta cartera son la **misma duración en distinta escala**: `exposicion = antiguedad / 10`. No son dos características independientes.

**Qué mirar:** el centro, la asimetría, los límites y las acumulaciones. Cambiar el ancho de los intervalos puede cambiar mucho la impresión visual.

In [ ]:

# Histograma de antigüedad con cortes en los años enteros.
ggplot(cartera, aes(x = antiguedad)) +
  geom_histogram(binwidth = 1, boundary = 0, fill = "#2c3e50", color = "white", alpha = 0.8) +
  labs(
    title = "Distribución de la Antigüedad de las Pólizas",
    x = "Antigüedad (años)",
    y = "Frecuencia (Número de Pólizas)"
  ) +
  scale_x_continuous(breaks = 0:max(ceiling(cartera$antiguedad))) +
  theme_minimal()


In [ ]:
library(ggplot2)

# Crear histograma de la variable exposicion
ggplot(cartera, aes(x = exposicion)) +
  geom_histogram(binwidth = 0.05, fill = "#16a085", color = "white", alpha = 0.8) +
  labs(
    title = "Distribución de la Exposición al Riesgo",
    x = "Exposición (décadas de póliza)",
    y = "Frecuencia (Número de Pólizas)"
  ) +
  theme_minimal()


## 3. ¿Edad y antigüedad parecen gaussianas?

🏵 **Una campana es una hipótesis visual que podemos contrastar.** Superpondremos dos curvas diferentes:

- **Densidad suavizada:** resume la forma de los valores observados mediante suavizado; no supone que sean normales.
- **Normal de referencia:** una gaussiana con la misma media y desviación típica de la muestra. Sirve de comparación, no demuestra un modelo generador.

El eje vertical pasa de recuento a **densidad de probabilidad**: el área total de las barras es uno. Así podemos comparar el histograma con las curvas. La altura de la densidad en un punto no es la probabilidad de ese punto; las probabilidades corresponden a áreas sobre intervalos.

Para una referencia normal usamos

$$f(x)=\frac{1}{s\sqrt{2\pi}}\exp\!\left[-\frac12\left(\frac{x-m}{s}\right)^2\right],$$

donde $m$ es la media y $s$ la desviación típica observadas. En R, `dnorm(x, mean = m, sd = s)` evalúa esa curva.

**Técnica:** histograma + densidad suavizada + normal, seguido de un gráfico Q–Q. [Referencia de histogramas](https://ggplot2.tidyverse.org/reference/geom_histogram.html).

In [ ]:
# La función devuelve un gráfico; cambia variable y ancho para explorar.
comparar_normal <- function(variable, ancho, frontera = 0) {
  stopifnot(length(variable) == 1L, variable %in% names(cartera),
            length(ancho) == 1L, is.finite(ancho), ancho > 0)
  valores <- cartera[[variable]]
  stopifnot(is.numeric(valores), length(valores) > 1L,
            all(is.finite(valores)), sd(valores) > 0)
  media <- mean(valores)
  desviacion <- sd(valores)
  dominio <- range(c(valores, media + c(-3, 3) * desviacion))

  ggplot(tibble(valor = valores), aes(valor)) +
    geom_histogram(aes(y = after_stat(density)), binwidth = ancho,
                   boundary = frontera, fill = "grey80", colour = "white") +
    geom_density(aes(colour = "Densidad suavizada"), linewidth = 0.8) +
    stat_function(fun = dnorm, args = list(mean = media, sd = desviacion),
                  aes(colour = "Normal de referencia"),
                  xlim = dominio, n = 401, linewidth = 0.9) +
    scale_colour_manual(values = c("Densidad suavizada" = "steelblue",
                                    "Normal de referencia" = "tomato")) +
    labs(title = paste("¿Se parece a una normal?", variable),
         subtitle = sprintf("Media = %.2f; desviación típica = %.2f", media, desviacion),
         x = paste(variable, "(años)"), y = "Densidad de probabilidad",
         colour = NULL) +
    theme(legend.position = "bottom")
}

print(comparar_normal("edad_conductor", ancho = 1, frontera = 17.5))
print(comparar_normal("antiguedad", ancho = 0.5, frontera = 0))

### Qué estamos viendo

- **Edad:** media **45,37** años y mediana **45**. El centro recuerda una campana, pero hay **109 personas de 18 años**, un límite inferior y valores enteros. La normal es continua y asigna masa fuera de esos límites. La acumulación en 18 es observable; el gráfico no establece por sí solo cómo se generó.
- **Antigüedad de la póliza:** media **4,21** años y mediana **3,768**, con rango **0,1–9,996**. La concentración hacia valores menores, la cola derecha y los límites contradicen una normal exacta.
- La curva suavizada depende de su suavizado y puede extenderse algo fuera de los límites. Es una ayuda visual, no una nueva observación.

**Prueba pequeña:** cambia el ancho de edad de `1` a `5`, o el de antigüedad de `0.5` a `1`. ¿La conclusión sobre la forma se mantiene?

⚜️ Un GLM de conteos no exige que sus variables explicativas sean normales. La familia del modelo se refiere a la **respuesta condicionada al perfil y a la exposición**. Todavía estamos describiendo los datos.

In [ ]:
cartera %>%
  select(edad_conductor, antiguedad) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "valor") %>%
  group_by(variable) %>%
  summarise(n = n(), media = mean(valor), mediana = median(valor),
            desviacion = sd(valor), minimo = min(valor), maximo = max(valor),
            .groups = "drop") %>% print()
cat("Conductores con 18 años:", sum(cartera$edad_conductor == 18), "\n")

### Q–Q: comparar posiciones en las distribuciones

Ordenamos los datos y comparamos sus cuantiles con los de una normal. Por ejemplo: ¿el percentil 10 observado ocupa la posición que esperaríamos bajo una forma normal?

- Una alineación aproximadamente recta indica una forma compatible con una normal en esa parte de la distribución.
- Curvas y desvíos en los extremos muestran dónde falla la aproximación.
- Los escalones de la edad reflejan que se registra en años enteros.

La recta de `stat_qq_line()` une los cuartiles de referencia. **Este gráfico no certifica normalidad**, y la inspección de su centro no sustituye mirar las colas. [Documentación de Q–Q](https://ggplot2.tidyverse.org/reference/geom_qq.html).

In [ ]:
qq <- cartera %>% select(antiguedad, edad_conductor) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "valor")
ggplot(qq, aes(sample = valor)) +
  stat_qq(alpha = 0.3, size = 0.9, colour = "steelblue") +
  stat_qq_line(colour = "tomato", linewidth = 0.8) +
  facet_wrap(~ variable, scales = "free_y") +
  labs(title = "Q–Q frente a una normal",
       x = "Cuantil normal teórico", y = "Cuantil observado (años)")

## 4. Una vista rápida de las demás medidas

**Técnica:** pequeños histogramas por variable. Las facetas tienen escalas propias porque las unidades son distintas: compara **formas**, no alturas entre paneles. Los nombres identifican las columnas del CSV; en la densidad de población el valor es habitantes/km², mientras que aquí el eje vertical cuenta pólizas.

Para edad y antigüedad del vehículo, que son enteras, una vista detallada con barras por año puede revelar acumulaciones que un intervalo grande escondería. El panel es un primer vistazo.

In [ ]:
medidas <- c("antiguedad", "edad_conductor", "antiguedad_carnet",
             "potencia_cv", "cilindrada", "antiguedad_vehiculo",
             "valor_vehiculo", "densidad", "km_anuales")
etiquetas <- c(
  antiguedad = "Póliza: antigüedad (años)",
  edad_conductor = "Edad (años)", antiguedad_carnet = "Carnet: antigüedad (años)",
  potencia_cv = "Potencia (CV)", cilindrada = "Cilindrada (unidad del CSV)",
  antiguedad_vehiculo = "Vehículo: antigüedad (años)",
  valor_vehiculo = "Valor del vehículo (u. monetarias)",
  densidad = "Densidad de población (hab/km²)", km_anuales = "Recorrido (km/año)"
)
cartera %>% select(all_of(medidas)) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "valor") %>%
  ggplot(aes(valor)) +
  geom_histogram(bins = 24, fill = "#287C8E", colour = "white") +
  facet_wrap(~ variable, scales = "free", ncol = 3,
             labeller = as_labeller(etiquetas)) +
  labs(x = NULL, y = "Pólizas", title = "Formas, límites y colas",
       subtitle = "Cada panel tiene su escala; todos los valores observados se conservan")

## 5. Conteos: conservar los enteros y los ceros

**Pregunta:** ¿cuántas pólizas han tenido 0, 1, 2… sucesos?

Una barra por entero evita que el histograma agrupe arbitrariamente conteos distintos. Hay **2113 de 3062 pólizas sin siniestros: aproximadamente el 69 %**. Esa proporción, por sí sola, no demuestra que necesitemos un modelo con inflación de ceros: las exposiciones y los perfiles también varían.

`n_conductores` cuenta personas cubiertas; se muestra aparte de los sucesos durante el seguimiento.

In [ ]:
conteos <- c("n_siniestros", "n_asistencia", "n_fraude", "n_gestiones")
largos <- cartera %>% select(all_of(conteos)) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "conteo")
ggplot(largos, aes(conteo)) +
  geom_bar(fill = "steelblue") +
  facet_wrap(~ variable, ncol = 2) +
  scale_x_continuous(breaks = seq(0, max(largos$conteo), by = 2)) +
  labs(x = "Sucesos durante el seguimiento", y = "Pólizas",
       subtitle = "Una barra por entero; mismos ejes para los cuatro conteos")

print(ggplot(cartera, aes(factor(n_conductores))) +
  geom_bar(fill = "steelblue") + labs(x = "Número de conductores", y = "Pólizas"))
cat("Pólizas sin siniestros:", sum(cartera$n_siniestros == 0), "de", nrow(cartera), "\n")

## 6. Categorías: tamaños y composición de la cartera

**Técnica:** barras. La longitud permite comparar tamaños; todos los ejes de recuento empiezan en cero. Primero vemos las categorías en facetas y después una función permite ampliar cualquiera con su recuento y porcentaje.

Las clases bonus-malus se mantienen en orden numérico; para estudios se usa básico → medio → superior. Las etiquetas originales de `edad_tramo` se conservan, aunque los rangos observados son 18–25, 26–35, 36–65 y 66–88: no deduzcas los límites exactos solo de sus nombres.

`id_poliza` se usa para consultar una fila. Región y agencia aparecen en el bloque jerárquico siguiente.

In [ ]:
categorias <- c(
  "edad_tramo", "zona_circulacion", "uso", "tipo_vehiculo", "combustible",
  "sexo", "color_vehiculo", "estado_civil", "forma_pago", "financiado",
  "nivel_estudios", "medio_contacto", "evento", "baja", "motivo_fin",
  "tuvo_siniestro", "bonus_malus_prev", "bonus_malus_act"
)
categorias_largas <- cartera %>%
  select(all_of(categorias)) %>% mutate(across(everything(), as.character)) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "nivel")

# Dos paneles para poder leer las etiquetas de todas las categorías.
for (grupo in split(categorias, ceiling(seq_along(categorias) / 9))) {
  print(ggplot(filter(categorias_largas, variable %in% grupo), aes(nivel)) +
    geom_bar(fill = "steelblue") +
    facet_wrap(~ variable, scales = "free", ncol = 3) + coord_flip() +
    labs(x = NULL, y = "Pólizas", subtitle = "Escalas propias por panel; leer sus valores"))
}

ver_categoria <- function(variable) {
  stopifnot(length(variable) == 1L, variable %in% categorias)
  niveles <- sort(unique(cartera[[variable]]))
  if (variable == "edad_tramo") niveles <- c("18-25", "25-35", "35-65", "65+")
  if (variable == "nivel_estudios") niveles <- c("basico", "medio", "superior")
  tabla <- cartera %>%
    transmute(nivel = factor(.data[[variable]], levels = niveles)) %>%
    count(nivel, .drop = FALSE) %>% mutate(porcentaje = 100 * n / sum(n))
  print(tabla)
  print(ggplot(tabla, aes(nivel, n)) + geom_col(fill = "steelblue") +
    coord_flip() + labs(title = variable, x = NULL, y = "Pólizas"))
  invisible(tabla)
}

# Cambia solo este nombre para ver una categoría con más detalle.
ver_categoria("nivel_estudios")

### Región y agencia: ver el anidamiento

El **treemap** dedica a cada agencia un área proporcional a sus pólizas; los bordes agrupan agencias por región. Es útil para ver composición y jerarquía. Las barras que lo acompañan permiten comparar los tamaños con más precisión.

El color identifica la región; no representa riesgo ni tasa de siniestros.

In [ ]:
agencias <- cartera %>% count(id_region, id_agencia, name = "polizas")
stopifnot(nrow(agencias) == n_distinct(cartera$id_agencia))
ggplot(agencias, aes(area = polizas, fill = factor(id_region),
                    subgroup = factor(id_region),
                    label = paste0("Ag. ", id_agencia, "\nn=", polizas))) +
  geom_treemap() + geom_treemap_subgroup_border(colour = "white", size = 2) +
  geom_treemap_text(colour = "white", place = "centre", min.size = 6) +
  labs(title = "Pólizas por región y agencia", fill = "Región")

print(ggplot(agencias, aes(factor(id_agencia), polizas, fill = factor(id_region))) +
  geom_col() + coord_flip() +
  labs(x = "Agencia", y = "Pólizas", fill = "Región"))

## 7. Relaciones: cada punto es una póliza

### Edad y antigüedad del carnet

**Técnica:** dispersión con transparencia. La transparencia revela dónde se superponen muchas observaciones.

La correlación lineal observada es **0,866**. Parte de la estructura está acotada por la edad: en esta cartera se cumple `antiguedad_carnet <= edad_conductor - 18`. La línea indica esa frontera. La correlación describe asociación; no establece causalidad.

In [ ]:
ggplot(cartera, aes(edad_conductor, antiguedad_carnet)) +
  geom_point(alpha = 0.2, size = 1, colour = "steelblue") +
  geom_abline(intercept = -18, slope = 1, colour = "tomato", linetype = 2) +
  labs(x = "Edad del conductor (años)", y = "Antigüedad del carnet (años)",
       subtitle = "Línea: antigüedad del carnet = edad − 18")
cat("Correlación:", cor(cartera$edad_conductor, cartera$antiguedad_carnet), "\n")

### Potencia y cilindrada

La correlación observada es **0,935**. Dos columnas muy asociadas pueden aportar información redundante a un futuro modelo. Eso es una pregunta para el modelado; aquí no eliminamos variables automáticamente.

**Si los puntos se pisan demasiado:** un recuento bidimensional agrupa el plano en celdas y usa el color para contar pólizas dentro de cada celda.

In [ ]:
ggplot(cartera, aes(potencia_cv, cilindrada)) +
  geom_point(alpha = 0.2, size = 1, colour = "steelblue") +
  labs(x = "Potencia (CV)", y = "Cilindrada (unidad del CSV)")

print(ggplot(cartera, aes(potencia_cv, cilindrada)) +
  geom_bin_2d(bins = 25) + scale_fill_viridis_c() +
  labs(x = "Potencia (CV)", y = "Cilindrada (unidad del CSV)", fill = "Pólizas"))
cat("Correlación:", cor(cartera$potencia_cv, cartera$cilindrada), "\n")

### Matriz de pares: una familia pequeña de variables

Una matriz de dispersión reúne varias parejas. Conservamos la idea de `plot(data)`, pero dejamos `id_poliza` fuera: su posición en la tabla no es una característica cuantitativa del riesgo.

Para estudiar el vehículo usamos cuatro medidas relacionadas. Una matriz de 40 × 40 sería difícil de leer e incluiría códigos y variables derivadas. `GGally::ggpairs()` es otra opción para añadir resúmenes a la diagonal; aquí usamos `pairs()` de R base, sin añadir una dependencia.

In [ ]:
pairs(cartera[c("potencia_cv", "cilindrada", "antiguedad_vehiculo", "valor_vehiculo")],
      labels = c("Potencia (CV)", "Cilindrada", "Edad vehículo", "Valor vehículo"),
      pch = 16, cex = 0.45, col = adjustcolor("steelblue", alpha.f = 0.2))

## 8. Comparar grupos sin perder su tamaño

### Cajas: kilómetros anuales por uso y combustible

La caja contiene la mitad central de los datos; su línea interior es la mediana. Los bigotes llegan hasta observaciones dentro de 1,5 rangos intercuartílicos desde los bordes de la caja. Los puntos que quedan fuera merecen inspección; no son automáticamente errores.

Consulta los tamaños de grupo antes de interpretar diferencias. Esta comparación es descriptiva y no ajusta por otros perfiles.

In [ ]:
cartera %>% count(uso, combustible) %>% print()
ggplot(cartera, aes(uso, km_anuales, fill = combustible)) +
  geom_boxplot(outlier.alpha = 0.25) +
  labs(x = "Uso", y = "Kilómetros anuales", fill = "Combustible") +
  theme(legend.position = "bottom")

### Tipo de vehículo y combustible: cantidad y proporción

Las barras agrupadas contestan **cuántas pólizas** hay de cada combinación. Las barras al 100 % contestan **cómo se reparte el combustible dentro de cada tipo**. El cambio de denominador cambia la pregunta: un grupo pequeño puede tener una proporción alta.

In [ ]:
cartera %>% count(tipo_vehiculo, combustible) %>% print()
ggplot(cartera, aes(tipo_vehiculo, fill = combustible)) +
  geom_bar(position = "dodge") +
  labs(x = "Tipo de vehículo", y = "Pólizas", fill = "Combustible")

print(ggplot(cartera, aes(tipo_vehiculo, fill = combustible)) +
  geom_bar(position = "fill") +
  scale_y_continuous(labels = scales::label_percent()) +
  labs(x = "Tipo de vehículo", y = "Porcentaje dentro de cada tipo", fill = "Combustible"))

### Facetas del carnet por tramo de edad

Los tramos contienen números distintos de pólizas. Usar densidad dentro de cada grupo permite comparar formas; mostrar `n` en el resumen conserva el tamaño. Se mantienen los mismos intervalos y ejes entre facetas.

In [ ]:
cartera %>% count(edad_tramo) %>% print()
ggplot(cartera, aes(antiguedad_carnet, y = after_stat(density))) +
  geom_histogram(binwidth = 5, boundary = 0, fill = "steelblue", colour = "white") +
  facet_wrap(~ factor(edad_tramo, levels = c("18-25", "25-35", "35-65", "65+"))) +
  labs(x = "Antigüedad del carnet (años)", y = "Densidad dentro de cada tramo")

### Escala logarítmica: densidad de población y zona

Una escala logarítmica permite leer valores separados por varios órdenes de magnitud. Distancias iguales representan cocientes iguales: pasar de 100 a 1000 ocupa lo mismo que pasar de 1000 a 10000.

Los rangos de densidad separan las zonas en esta muestra. Esa separación puede reflejar cómo se construyó la variable; no constituye evidencia de un efecto causal. El gráfico conserva los valores originales y transforma la escala del eje.

In [ ]:
stopifnot(all(cartera$densidad > 0))
ggplot(cartera, aes(zona_circulacion, densidad)) +
  geom_boxplot(fill = "grey85", outlier.alpha = 0.3) + scale_y_log10() +
  labs(x = "Zona de circulación", y = "Densidad de población (hab/km²; escala log10)")

## 9. Costes: separar cero, positivo y no definido

El coste total combina una masa en cero con costes positivos que presentan una cola derecha larga. Resumirlo todo con una única campana ocultaría esa estructura.

Primero contamos cada caso; después representamos los **costes positivos** en escala logarítmica. Los 2113 ceros siguen contabilizados, aunque no puedan situarse en un eje logarítmico. `coste_medio` corresponde al coste por siniestro de cada póliza con al menos un siniestro.

In [ ]:
cartera %>%
  summarise(polizas = n(), coste_total_cero = sum(coste_total == 0),
            coste_total_positivo = sum(coste_total > 0),
            coste_medio_definido = sum(!is.na(coste_medio))) %>% print()

positivos <- cartera %>% filter(coste_total > 0)
ggplot(positivos, aes(coste_total)) +
  geom_histogram(bins = 30, fill = "steelblue", colour = "white") +
  scale_x_log10() +
  labs(x = "Coste total positivo (unidad monetaria del CSV; escala log10)",
       y = "Pólizas", subtitle = paste("Solo costes positivos; n =", nrow(positivos)))

print(ggplot(positivos, aes(coste_medio)) +
  geom_histogram(bins = 30, fill = "tomato", colour = "white") +
  scale_x_log10() +
  labs(x = "Coste medio por siniestro (unidad monetaria del CSV; escala log10)",
       y = "Pólizas con siniestros"))

## 10. Bonus-malus: una transición en lugar de dos fotografías

**Técnica:** mapa de calor. La fila corresponde a la clase anterior y la columna a la actual. La diagonal reúne las pólizas que permanecen en la misma clase.

El número escrito es el **recuento de pólizas**. El color representa el **porcentaje dentro de cada clase anterior**: cada fila suma el 100 %. Así podemos comparar transiciones de clases que contienen cantidades distintas de pólizas.

In [ ]:
transiciones <- cartera %>% count(bonus_malus_prev, bonus_malus_act) %>%
  complete(bonus_malus_prev = 1:5, bonus_malus_act = 1:5, fill = list(n = 0)) %>%
  group_by(bonus_malus_prev) %>%
  mutate(porcentaje_fila = 100 * n / sum(n)) %>% ungroup()
ggplot(transiciones, aes(factor(bonus_malus_act), factor(bonus_malus_prev),
                         fill = porcentaje_fila)) +
  geom_tile(colour = "white") + geom_text(aes(label = n), colour = "black") +
  scale_fill_gradient(low = "white", high = "skyblue3", limits = c(0, 100)) +
  coord_equal() +
  labs(x = "Clase actual", y = "Clase anterior", fill = "% de la fila",
       subtitle = "Números: pólizas; color: porcentaje condicionado a la clase anterior")

## 11. Relojes: tiempo registrado y estado observado

Cuando `evento = 0`, `t_primer_sin` recoge el tiempo hasta el final de la observación sin un primer siniestro. Es **censura**, no un siniestro ocurrido al terminar el seguimiento.

La pareja `t_baja`–`baja` necesita la misma lectura conjunta. Conservamos la unidad y el origen de `t_baja` como pendientes de confirmar; no lo intercambiamos con la antigüedad de la póliza.

Estos histogramas describen los tiempos registrados por estado. **No estiman una curva de supervivencia**: las observaciones censuradas requieren un tratamiento específico para hacerlo.

In [ ]:
# CELDA 3. Cada reloj se lee con su indicador. No se comparte escala entre ellos.
cartera %>%
  mutate(estado = if_else(evento == 1, "Siniestro observado", "Censura")) %>%
  ggplot(aes(t_primer_sin)) +
  geom_histogram(binwidth = 0.5, boundary = 0, fill = "#287C8E", colour = "white") +
  facet_wrap(~ estado, ncol = 1) +
  labs(x = "Tiempo hasta primer siniestro o censura (años)", y = "Pólizas",
       subtitle = "Duraciones registradas; esto no estima la supervivencia") +
  theme_minimal()

cartera %>%
  mutate(estado = if_else(baja == 1, "Baja observada", "Censura")) %>%
  ggplot(aes(t_baja)) +
  geom_histogram(bins = 20, fill = "#287C8E", colour = "white") +
  facet_wrap(~ estado, ncol = 1) +
  labs(x = "t_baja (unidad y origen por confirmar)", y = "Pólizas",
       subtitle = "Se conserva el registro original; leer junto al indicador baja") +
  theme_minimal()
# t_baja y antiguedad no son intercambiables: confirmar sus relojes.

## 12. Qué relaciones vienen de una definición

Una correlación perfecta puede ser una duplicación de información. Las siguientes comprobaciones verifican relaciones observadas en esta cartera; no las convierten automáticamente en reglas de cualquier otro dataset.

- `exposicion` es la antigüedad dividida entre diez.
- `evento` y `tuvo_siniestro` coinciden con tener al menos un siniestro.
- `coste_medio` coincide con coste total / siniestros, salvo redondeo a dos decimales.

Estas relaciones ayudan a no interpretar como descubrimiento lo que ya está contenido en la definición de las variables.

In [ ]:
tibble(
  comprobacion = c("exposicion = antiguedad/10",
                  "evento = tuvo_siniestro",
                  "tuvo_siniestro = (n_siniestros > 0)",
                  "coste_medio ausente equivale a siniestros = 0"),
  se_cumple = c(
    isTRUE(all.equal(cartera$exposicion, cartera$antiguedad / 10)),
    all(cartera$evento == cartera$tuvo_siniestro),
    all(cartera$tuvo_siniestro == as.integer(cartera$n_siniestros > 0)),
    all(is.na(cartera$coste_medio) == (cartera$n_siniestros == 0))
  )
)

## 13. De contar siniestros a medir su ritmo

🔰 Conservamos tus nombres y separamos las dos cantidades:

- $y[i]$: siniestros observados en la póliza $i$.
- $X[i]$: exposición observada, expresada en años.
- $r[i] = y[i]/X[i]$: **tasa observada** de esa póliza.
- $\lambda[i]$: **tasa esperada**, que un modelo estimará a partir del perfil.
- $\mu[i] = E[Y_i\mid\text{perfil},X_i] = X[i]\,\lambda[i]$: número esperado durante su exposición, bajo el supuesto de proporcionalidad.

⚜️ Multiplicar $X[i]$ por $r[i]$ devuelve exactamente $y[i]$: es reconstruir el dato. Para predecir necesitamos estimar $\lambda[i]$ usando un modelo y sus supuestos.

`tasa()` conserva la operación vectorizada que ya usabas. Sus comprobaciones evitan reciclar vectores de longitudes distintas, dividir por cero o tratar una ausencia como si fuera un dato válido.

In [ ]:
# Función de cálculo de tasas observadas, con control de errores.
tasa <- function(eventos, exposicion) {
  stopifnot(
    is.numeric(eventos), is.numeric(exposicion),
    length(eventos) == length(exposicion),
    all(is.finite(eventos)), all(eventos >= 0),
    all(is.finite(exposicion)), all(exposicion > 0)
  )
  eventos / exposicion
}

In [ ]:
#-- Datos de interés
data <- cartera %>%
    mutate(
        exposicion_anual = exposicion * 10,
        tasa_siniestros  = tasa(
                n_siniestros,
                exposicion_anual)
        ) %>%
    select(
        id_poliza,
        exposicion_anual,
        n_siniestros,
        tasa_siniestros
        )

#-- Comprobación
head(data)

### Tasa global: dividir los totales

Un año-póliza es una póliza observada durante un año; dos pólizas observadas medio año suman también un año-póliza.

$$\lambda_{\mathrm{global}} = \frac{\sum_i y[i]}{\sum_i X[i]}
  = \frac{1770}{12882{,}354}
  \approx 0{,}1374\ \text{siniestros por año-póliza}.$$

Son aproximadamente **13,74 siniestros por 100 años-póliza**. Esta cantidad es una tasa, no una probabilidad expresada en porcentaje.

`mean(data$tasa_siniestros)` asigna el mismo peso a cada póliza; el cociente de totales equivale a ponderar cada tasa individual por su exposición. Si ambas medias usan exactamente las mismas filas completas, `mean(y) / mean(X)` también coincide con `sum(y) / sum(X)`, porque los dos tamaños muestrales se cancelan.

In [ ]:
tasa_total <- sum(data$n_siniestros) / sum(data$exposicion_anual)
tasa_total

tibble(
  siniestros = sum(data$n_siniestros),
  anos_poliza = sum(data$exposicion_anual),
  tasa_global = tasa_total,
  media_de_tasas = mean(data$tasa_siniestros),
  media_ponderada = weighted.mean(data$tasa_siniestros, data$exposicion_anual)
)

In [ ]:
summary(data)

### Exposición frente a siniestros: empezar por los puntos

**Pregunta:** ¿qué conteos observamos según los años de exposición?

El eje horizontal contiene años y el vertical conteos enteros. Desplazamos ligeramente los puntos **solo en vertical y solo para dibujar**, de modo que se puedan distinguir pólizas con el mismo conteo. Los datos no se modifican.

La línea representa `exposición × tasa global`: una referencia de tasa constante para toda la cartera. No es un modelo ajustado por perfil ni una predicción individual validada.

In [ ]:
ggplot(data, aes(exposicion_anual, n_siniestros)) +
  geom_point(position = position_jitter(width = 0, height = 0.12, seed = 7),
             alpha = 0.18, size = 1, colour = "steelblue") +
  geom_abline(intercept = 0, slope = tasa_total, colour = "tomato", linewidth = 0.8) +
  scale_y_continuous(breaks = 0:max(data$n_siniestros)) +
  labs(x = "Exposición (años)", y = "Número de siniestros",
       subtitle = "Línea: exposición × tasa global; puntos con desplazamiento visual")

### El violín responde a la pregunta inversa

**Pregunta:** entre las pólizas con $N$ siniestros, ¿cómo se distribuyen sus años de exposición?

Por eso agrupamos el conteo en el eje horizontal y dibujamos la variable continua —la exposición— en el vertical. La anchura del violín indica concentración relativa dentro del grupo; los puntos muestran los registros y el punto negro marca la mediana.

Todos los violines tienen la misma anchura máxima, así que **su anchura no compara tamaños entre grupos**. El tamaño `n` se escribe en cada etiqueta. Solo suavizamos grupos con al menos 10 pólizas; es un criterio gráfico explícito, no un umbral de validez estadística. Los grupos pequeños conservan sus puntos y mediana.

Este gráfico describe $X\mid N$. No responde directamente a la predicción de $N\mid X$. [Documentación del violín](https://ggplot2.tidyverse.org/reference/geom_violin.html).

In [ ]:
grupos_siniestros <- data %>% count(n_siniestros, name = "polizas") %>%
  arrange(n_siniestros)
datos_violin <- data %>% left_join(grupos_siniestros, by = "n_siniestros") %>%
  mutate(grupo = factor(n_siniestros, levels = grupos_siniestros$n_siniestros))
etiquetas <- setNames(
  paste0(grupos_siniestros$n_siniestros, "\nn=", grupos_siniestros$polizas),
  as.character(grupos_siniestros$n_siniestros)
)
ggplot(datos_violin, aes(grupo, exposicion_anual)) +
  geom_violin(data = filter(datos_violin, polizas >= 10),
              scale = "width", trim = TRUE, fill = "skyblue", alpha = 0.45) +
  geom_point(position = position_jitter(width = 0.12, height = 0, seed = 7),
             alpha = 0.12, size = 0.8, colour = "steelblue") +
  stat_summary(fun = median, geom = "point", colour = "black", size = 2) +
  scale_x_discrete(labels = etiquetas, drop = FALSE) +
  labs(x = "Número de siniestros y tamaño del grupo", y = "Exposición (años)")

### Tasas por tipo de vehículo

Comparamos grupos sumando siniestros y exposición **dentro del mismo grupo**. Mostrar también pólizas, siniestros y años-póliza deja visible de dónde sale la tasa.

Las diferencias son descriptivas: pueden intervenir edad, uso, exposición y otras características. Este gráfico no estima un efecto causal ni incluye intervalos de incertidumbre.

In [ ]:
tasas_tipo <- cartera %>% group_by(tipo_vehiculo) %>%
  summarise(polizas = n(), siniestros = sum(n_siniestros),
            anos_poliza = sum(exposicion * 10), .groups = "drop") %>%
  mutate(tasa = siniestros / anos_poliza)
print(tasas_tipo)
ggplot(tasas_tipo, aes(tipo_vehiculo, tasa)) +
  geom_point(size = 3, colour = "steelblue") +
  geom_hline(yintercept = tasa_total, linetype = 2, colour = "tomato") +
  labs(x = "Tipo de vehículo", y = "Siniestros por año-póliza",
       subtitle = "Suma de siniestros / suma de años; línea: tasa global")

### Tu matriz de tasas: interpretar las relaciones derivadas

Se conserva `plot(data)` como una matriz de pares, excluyendo únicamente el identificador. Recuerda que `tasa_siniestros = n_siniestros / exposicion_anual`: sus asociaciones no son tres descubrimientos independientes.

In [ ]:
plot(data[c("exposicion_anual", "n_siniestros", "tasa_siniestros")],
     col = adjustcolor("tomato", alpha.f = 0.2), pch = 16, cex = 0.5)

## 14. Puente hacia el GLM: qué aportará el modelo

Bajo el supuesto de una tasa esperada constante durante la exposición de cada póliza:

$$\mu_i = X_i\lambda_i.$$

Aplicar una **transformación logarítmica** a esa media positiva convierte el producto en una suma:

$$\log\mu_i = \log X_i + \log\lambda_i.$$

Si modelamos el logaritmo de la tasa como una combinación lineal del perfil,

$$\log\lambda_i = \beta_0 + \beta_1x_{i1} + \cdots,$$

obtenemos

$$\log\mu_i = \underbrace{\log X_i}_{\text{offset}} + \beta_0 + \beta_1x_{i1} + \cdots.$$

El coeficiente del offset está fijado en uno: manteniendo el perfil y la tasa, duplicar la exposición duplica la media esperada. **No estamos aplicando `log()` al conteo observado**, que puede ser cero. Tampoco hemos ajustado todavía un modelo: la exploración sirve para saber qué describen sus variables y qué supuestos habrá que revisar.

La normal de referencia que dibujamos para la edad no elige la familia de ese GLM. Una futura familia Poisson o binomial negativa describirá la variación del conteo alrededor de su media condicionada al perfil y al tiempo.

## 15. Qué hemos aprendido a mirar

- **Histogramas:** dónde se concentra una medida y cómo influyen los intervalos.
- **Curva normal y Q–Q:** dónde una aproximación normal describe bien o mal la forma.
- **Barras:** cuántos registros hay por categoría o conteo, incluidos los ceros.
- **Dispersión y recuentos 2D:** asociación, límites y superposición de observaciones.
- **Matriz de pares:** varias relaciones dentro de una familia pequeña de medidas.
- **Cajas y violines:** distribución de una medida dentro de grupos; leer siempre sus tamaños.
- **Treemap y mapa de calor:** jerarquía de agencias y transición entre clases.
- **Escala logarítmica:** comparar órdenes de magnitud, contabilizando antes los ceros.
- **Tasas:** distinguir cantidad acumulada y ritmo por unidad de exposición.

**Para continuar juntos:** elige una figura y escribe qué representa cada eje, qué observación puedes defender y qué pregunta queda abierta. Ese paso conecta la exploración con un futuro modelo sin dar un salto de fe.

### Fuentes y reproducibilidad

- [Cartera fijada del equipo 07-Cox](https://raw.githubusercontent.com/JulTob/Modelling/698dad06c4228ea79319fdba27647a0c61474703/GLM/cartera.csv).
- [Material de la profesora: caso de conteos](https://asunmayoral.github.io/GLM/caso2/caso2_conteos.html). Los datos de referencia de la profesora y los de cada equipo pueden diferir.
- [Histogramas](https://ggplot2.tidyverse.org/reference/geom_histogram.html), [Q–Q](https://ggplot2.tidyverse.org/reference/geom_qq.html), [barras](https://ggplot2.tidyverse.org/reference/geom_bar.html), [cajas](https://ggplot2.tidyverse.org/reference/geom_boxplot.html), [violines](https://ggplot2.tidyverse.org/reference/geom_violin.html) y [recuentos 2D](https://ggplot2.tidyverse.org/reference/geom_bin_2d.html): documentación oficial de ggplot2.
- [Distribución normal en R](https://stat.ethz.ch/R-manual/R-release/library/stats/html/Normal.html) y [matrices de pares](https://stat.ethz.ch/R-manual/R-release/library/graphics/html/pairs.html): documentación de R.
- [Treemap y agrupación jerárquica](https://wilkox.org/treemapify/): documentación de treemapify.

Las cifras proceden del CSV fijado y se reproducen en las celdas. Las interpretaciones se distinguen de las comprobaciones algebraicas. Las unidades monetarias, la unidad documental de cilindrada y el origen/unidad de `t_baja` quedan expresamente pendientes de confirmar.